# Level 2 Task 3: Clustering Analysis (K-Means)

This notebook applies K-Means clustering to the housing dataset after standardizing selected numerical features.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "data" / "house_prediction_dataset.csv").exists():
    base_dir = Path("Level_2_Intermediate") / "Task_3_Clustering_Analysis"

data_path = base_dir / "data" / "house_prediction_dataset.csv"
output_dir = base_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

column_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
features = ["RM", "LSTAT", "MEDV", "CRIM", "NOX", "DIS"]
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(data_path, sep=r"\s+", header=None, names=column_names)
print("Dataset shape:", df.shape)
df.head()

## Standardize Features

In [ ]:
means = df[features].mean()
stds = df[features].std(ddof=0)
scaled_df = (df[features] - means) / stds
scaled_data = scaled_df.to_numpy()
scaled_df.head()

## Define K-Means

In [ ]:
def kmeans(data, k, random_state=42, max_iter=100):
    rng = np.random.default_rng(random_state)
    centers = data[rng.choice(len(data), size=k, replace=False)].copy()
    for _ in range(max_iter):
        distances = ((data[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = distances.argmin(axis=1)
        new_centers = np.array([
            data[labels == cluster_id].mean(axis=0) if np.any(labels == cluster_id) else centers[cluster_id]
            for cluster_id in range(k)
        ])
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    inertia = ((data - centers[labels]) ** 2).sum()
    return labels, centers, inertia

## Elbow Method

In [ ]:
elbow_records = []
for k in range(2, 7):
    _, _, inertia = kmeans(scaled_data, k=k, random_state=42)
    elbow_records.append({"k": k, "inertia": round(float(inertia), 3)})
elbow_df = pd.DataFrame(elbow_records)
elbow_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(elbow_df["k"], elbow_df["inertia"], marker="o", linewidth=2.2, color="#1f77b4")
plt.title("Elbow Method for K-Means", fontweight="bold")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(elbow_df["k"])
plt.tight_layout()
plt.show()

## Fit Final Model

In [ ]:
chosen_k = 3
labels, centers, inertia = kmeans(scaled_data, k=chosen_k, random_state=42)
df["cluster"] = labels
print("Chosen k:", chosen_k)
print("Inertia:", round(float(inertia), 3))
df[["RM", "LSTAT", "MEDV", "cluster"]].head()

In [ ]:
cluster_profiles = df.groupby("cluster")[features].mean().round(3)
cluster_profiles

## Cluster Visualization

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="RM", y="LSTAT", hue="cluster", palette="Set2", s=70, alpha=0.9)
plt.title("K-Means Clusters in RM vs LSTAT Space", fontweight="bold")
plt.xlabel("Average Number of Rooms (RM)")
plt.ylabel("Lower Status Population Percentage (LSTAT)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()